## Parte 2. Análisis de GLOBE con datos ambientales

Si todavía no has utilizado Google Earth Engine, sigue [este tutorial](https://geo-di-lab.github.io/emerge-lessons/docs/ch1/lesson5a.html) para configurar tu proyecto de Earth Engine. Esto te permitirá cargar datos ambientales para tu análisis de GLOBE.

## Introducción a las herramientas

* **Python** es un lenguaje de programación popular que se utiliza para analizar datos, automatizar tareas, crear software, entrenar modelos de aprendizaje automático y mucho más. Lo utilizaremos para analizar datos de GLOBE Mosquito Habitat Mapper y GLOBE Land Cover.
* **Google Colab** es un entorno gratuito de programación en la nube que permite ejecutar Python desde el navegador. *¡No es necesario instalar ni descargar nada!*
* **Google Earth Engine** es una plataforma en la nube que incluye imágenes satelitales y otros conjuntos de datos geoespaciales. La utilizaremos para cargar datos ambientales.

**Para ejecutar el código:**

Cada bloque de código se denomina celda. Para ejecutar una celda, pasa el cursor sobre ella y haz clic en la flecha de la esquina superior izquierda, o haz clic dentro de la celda y presiona `Shift + Enter`.

Nota: La primera vez que ejecutes un bloque de código, Google Colab mostrará el mensaje `Warning: This notebook was not authored by Google`. Haz clic en `Run Anyway`.

In [ ]:
# Escribe a continuación el ID de tu proyecto de Google Earth Engine
projectID = "Write Your Project ID"

# Por ejemplo:
# projectID = "emerge-lessons-23148" 

In [ ]:
# Escribe los nombres del estado y del condado en inglés,
# tal como aparecen en los datos del Censo de Estados Unidos
state_name = "Your State Name"
county_name = "Your County Name"

# Por ejemplo:
# state_name = "Florida"
# county_name = "Broward County"

# El nombre del condado debe terminar con la palabra "County" 

In [ ]:
# Importar las bibliotecas
import pandas as pd                           # Trabajar con datos
pd.set_option("display.max_columns", None)    # Mostrar todas las columnas
import geopandas as gpd                       # Trabajar con datos espaciales
import numpy as np                            # Trabajar con números
from datetime import date                     # Trabajar con fechas
import matplotlib.pyplot as plt               # Crear gráficos
from matplotlib.colors import to_rgb          # Obtener colores
import branca.colormap as cm                  # Crear escalas de colores
import seaborn as sns                         # Usar paletas y crear gráficos
import folium                                 # Crear mapas interactivos
import ee                                     # Cargar datos de Earth Engine
import geemap                                 # Crear mapas de Earth Engine

# Inicializar Earth Engine para cargar los datos ambientales
ee.Authenticate()
ee.Initialize(project=projectID)

In [ ]:
# Cargar los datos
mosquito = gpd.read_file(
    "https://github.com/geo-di-lab/emerge-lessons/"
    "raw/refs/heads/main/docs/data/globe_mosquito.zip"
)
land_cover = gpd.read_file(
    "https://github.com/geo-di-lab/emerge-lessons/"
    "raw/refs/heads/main/docs/data/globe_land_cover.zip"
)
counties = gpd.read_file(
    "https://github.com/geo-di-lab/emerge-lessons/"
    "raw/refs/heads/main/docs/data/us_counties.zip"
).to_crs("EPSG:4326")

# Obtener el límite del condado seleccionado
county = counties.loc[
    (counties["STATE_NAME"] == state_name)
    & (counties["NAMELSAD"] == county_name)
]

In [ ]:
# Convertir los límites del condado al formato de Earth Engine
region = geemap.geopandas_to_ee(county)

# Crear un mapa vacío centrado en el condado
county_center = county.iloc[0].geometry.centroid

map = geemap.Map(
    center=[county_center.y, county_center.x],
    zoom=10
)
map.add_basemap("CartoDB.DarkMatter")

Ahora representaremos en un mapa la temperatura de la superficie terrestre de todo el condado. Primero, define el intervalo de fechas que se utilizará para calcular la temperatura promedio.

In [ ]:
start_date = "2025-06-01"
end_date = "2025-07-01" 

Visualizaremos datos de teledetección de NASA disponibles mediante Google Earth Engine: [MOD11A1.061 Terra Land Surface Temperature and Emissivity Daily Global 1km](https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MOD11A1).

Los valores de temperatura de la superficie terrestre están escalados. Para recuperar la temperatura original en kelvin, el valor debe multiplicarse por 0.02. Después podemos convertirlo a grados Fahrenheit.

In [ ]:
def to_fahrenheit(lst):
    """Convierte un valor escalado de MODIS LST a grados Fahrenheit."""
    celsius = lst * 0.02 - 273.15
    fahrenheit = celsius * 1.8 + 32
    return fahrenheit


def to_lst(fahrenheit):
    """Convierte grados Fahrenheit al valor escalado de MODIS LST."""
    celsius = (fahrenheit - 32) / 1.8
    lst = (celsius + 273.15) / 0.02
    return lst

In [ ]:
# Cargar los datos de Google Earth Engine
lst = (
    ee.ImageCollection("MODIS/061/MOD11A1")
    .filterDate(start_date, end_date)
    .select("LST_Day_1km")
    .median()
    .clip(region)
)

# Definir la escala de colores
colors = [
    "040274", "040281", "0502a3", "0502b8", "0502ce", "0502e6",
    "0602ff", "235cb1", "307ef3", "269db1", "30c8e2", "32d3ef",
    "3be285", "3ff38f", "86e26f", "3ae237", "b5e22e", "d6e21f",
    "fff705", "ffd611", "ffb613", "ff8b13", "ff6e08", "ff500d",
    "ff0000", "de0101", "c21301", "a71001", "911003"
]

# Definir los valores mínimo y máximo y la escala de colores
lst_vis = {
    "min": to_lst(50),
    "max": to_lst(100),
    "palette": colors
}

# Agregar los datos al mapa
map.addLayer(
    lst,
    lst_vis,
    "Temperatura de la superficie terrestre"
)

display(map)

La siguiente barra de colores muestra qué representan los colores del mapa: la temperatura de la superficie terrestre en grados Fahrenheit.

In [ ]:
plt.figure(figsize=(len(colors), 1))
plt.imshow([[to_rgb(f"#{color}") for color in colors]])

plt.text(
    -0.6,
    0.1,
    "50 °F",
    va="center",
    ha="right",
    fontsize=24
)
plt.text(
    len(colors) - 0.4,
    0.1,
    "100 °F",
    va="center",
    ha="left",
    fontsize=24
)

plt.axis("off")
plt.show()

A continuación, calcularemos la temperatura promedio de todo el condado.

In [ ]:
mean_lst = (
    lst.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=1000,
        maxPixels=1e12
    )
    .get("LST_Day_1km")
    .getInfo()
)

print(
    f"La temperatura promedio de {county_name} entre "
    f"{start_date} y {end_date} es "
    f"{round(to_fahrenheit(mean_lst), 2)} °F."
)

Ahora visualizaremos el uso y la cobertura del suelo del condado mediante el conjunto de datos [USFS Landscape Change Monitoring System v2024.10 (CONUS and OCONUS)](https://developers.google.com/earth-engine/datasets/catalog/USFS_GTAC_LCMS_v2024-10#bands).

In [ ]:
# Uso del suelo

# Reiniciar el mapa
map = geemap.Map(
    center=[county_center.y, county_center.x],
    zoom=10
)
map.add_basemap("CartoDB.DarkMatter")

# Definir la paleta de colores del uso del suelo
palette = [
    "fbff97",
    "e6558b",
    "004e2b",
    "9dbac5",
    "a6976a",
    "1b1716"
]
visual = {
    "min": 1,
    "max": 6,
    "palette": palette
}

# Agregar el uso del suelo de 1985
landcover_1985 = (
    ee.ImageCollection("USFS/GTAC/LCMS/v2024-10")
    .filterDate("1985", "1986")
    .filter('study_area == "CONUS"')
    .first()
    .clip(region)
)

map.addLayer(
    landcover_1985.select("Land_Use"),
    visual,
    "Uso del suelo en 1985"
)

# Agregar el uso del suelo de 2024
landcover_2024 = (
    ee.ImageCollection("USFS/GTAC/LCMS/v2024-10")
    .filterDate("2024", "2025")
    .filter('study_area == "CONUS"')
    .first()
    .clip(region)
)

map.addLayer(
    landcover_2024.select("Land_Use"),
    visual,
    "Uso del suelo en 2024"
)

display(map)

$$\color{#fbff97}Agricultura$$
$$\color{#e6558b}Desarrollo\ urbano$$
$$\color{#004e2b}Bosque$$
$$\color{#9dbac5}Otro$$
$$\color{#a6976a}Pastizal\ o\ terreno\ de\ pastoreo$$

Crearemos una función que calcule el porcentaje de cada tipo de uso y cobertura del suelo presente en el condado.

In [ ]:
def land_stats(image, name, labels):
    """
    Calcula el porcentaje de cada tipo de uso o cobertura del suelo
    dentro del condado.

    Argumentos:
        image: imagen de Earth Engine
        name: nombre de la banda, Land_Use o Land_Cover
        labels: diccionario de etiquetas
    """
    count = (
        image.select(name)
        .reduceRegion(
            reducer=ee.Reducer.frequencyHistogram(),
            geometry=region,
            scale=30,
            maxPixels=1e12
        )
        .getInfo()
    )

    land_cover_counts = {}

    for key, value in labels.items():
        if key in count.get(name, {}):
            land_cover_counts[value] = count[name][key]

    total_pixels = sum(land_cover_counts.values())

    if total_pixels == 0:
        print("No se encontraron píxeles válidos en la región.")
        return

    for label, num in land_cover_counts.items():
        percent = round(100 * num / total_pixels, 1)
        if percent > 0:
            print(f"{label}: {percent} %")

Obtén los porcentajes de uso del suelo correspondientes a 1985.

In [ ]:
land_use_labels = {
    "1": "Agricultura",
    "2": "Desarrollo urbano",
    "3": "Bosque",
    "4": "Otro",
    "5": "Pastizal o terreno de pastoreo",
    "6": "Máscara de área no procesada"
}

land_stats(
    landcover_1985,
    "Land_Use",
    land_use_labels
)

Ahora repetiremos el cálculo para 2024.

In [ ]:
land_stats(
    landcover_2024,
    "Land_Use",
    land_use_labels
)

Visualiza la cobertura terrestre en un mapa.

In [ ]:
# Cobertura terrestre

# Reiniciar el mapa
map = geemap.Map(
    center=[county_center.y, county_center.x],
    zoom=10
)
map.add_basemap("CartoDB.DarkMatter")

# Definir la paleta de colores
palette = [
    "004e2b", "009344", "61bb46", "acbb67", "8b8560",
    "cafd4b", "f89a1c", "8fa55f", "bebb8e", "e5e98a",
    "ddb925", "893f54", "e4f5fd", "00b6f0", "1b1716"
]
visual = {
    "min": 1,
    "max": 15,
    "palette": palette
}

# Agregar la cobertura terrestre de 1985 y 2024
map.addLayer(
    landcover_1985.select("Land_Cover"),
    visual,
    "Cobertura terrestre en 1985"
)
map.addLayer(
    landcover_2024.select("Land_Cover"),
    visual,
    "Cobertura terrestre en 2024"
)

display(map)

$$\color{#004e2b}Árboles$$
$$\color{#61bb46}Mezcla\ de\ arbustos\ y\ árboles$$
$$\color{#acbb67}Mezcla\ de\ pastos,\ hierbas\ y\ árboles$$
$$\color{#8b8560}Mezcla\ de\ terreno\ sin\ vegetación\ y\ árboles$$
$$\color{#f89a1c}Arbustos$$
$$\color{#8fa55f}Mezcla\ de\ pastos,\ hierbas\ y\ arbustos$$
$$\color{#bebb8e}Mezcla\ de\ terreno\ sin\ vegetación\ y\ arbustos$$
$$\color{#e5e98a}Pastos\ y\ hierbas$$
$$\color{#ddb925}Mezcla\ de\ terreno\ sin\ vegetación,\ pastos\ y\ hierbas$$
$$\color{#893f54}Terreno\ sin\ vegetación\ o\ impermeable$$
$$\color{#e4f5fd}Nieve\ o\ hielo$$

Calcula el porcentaje de cada tipo de cobertura terrestre.

In [ ]:
land_cover_labels = {
    "1": "Árboles",
    "2": "Mezcla de arbustos altos y árboles (solo Alaska)",
    "3": "Mezcla de arbustos y árboles",
    "4": "Mezcla de pastos, hierbas y árboles",
    "5": "Mezcla de terreno sin vegetación y árboles",
    "6": "Arbustos altos (solo Alaska)",
    "7": "Arbustos",
    "8": "Mezcla de pastos, hierbas y arbustos",
    "9": "Mezcla de terreno sin vegetación y arbustos",
    "10": "Pastos y hierbas",
    "11": "Mezcla de terreno sin vegetación, pastos y hierbas",
    "12": "Terreno sin vegetación o impermeable",
    "13": "Nieve o hielo",
    "14": "Agua",
    "15": "Máscara de área no procesada"
}

In [ ]:
land_stats(
    landcover_1985,
    "Land_Cover",
    land_cover_labels
)

In [ ]:
land_stats(
    landcover_2024,
    "Land_Cover",
    land_cover_labels
)

## Integración de los análisis: obtener datos de teledetección para un punto de GLOBE

Consulta la lista de observaciones del condado seleccionado.

In [ ]:
# Mostrar las observaciones de mosquitos del condado elegido
data_county = mosquito.sjoin(
    county,
    how="inner",
    predicate="within"
)
data_county

Elige una de estas observaciones y anota su índice, es decir, el número situado en la columna izquierda, como `6643` o `29024`. Escríbelo en la siguiente celda.

In [ ]:
# Sustituye None por el índice de la observación elegida
index = None

# Por ejemplo:
# index = 29024

Muestra los datos correspondientes al punto con ese índice:

In [ ]:
if index is None:
    raise ValueError(
        "Sustituye None por el índice de una observación "
        "antes de ejecutar esta celda."
    )

point = data_county[data_county.index == index]

if point.empty:
    raise ValueError(
        f"No se encontró una observación con el índice {index}."
    )

point

Obtendremos datos ambientales de temperatura de la superficie terrestre, precipitación y vegetación para el punto seleccionado. Puedes consultar más información sobre este código en el [capítulo 4 del libro digital](https://geo-di-lab.github.io/emerge-lessons/docs/ch4/lesson2.html).

In [ ]:
def mask_s2_clouds(image):
    """
    Enmascara las nubes de una imagen Sentinel-2 mediante la banda QA.

    Argumentos:
        image (ee.Image): imagen de Sentinel-2

    Devuelve:
        ee.Image: imagen de Sentinel-2 con las nubes enmascaradas
    """
    qa = image.select("QA60")

    # Los bits 10 y 11 representan nubes y cirros, respectivamente
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    # Ambas señales deben ser cero para indicar condiciones despejadas
    mask = (
        qa.bitwiseAnd(cloud_bit_mask)
        .eq(0)
        .And(
            qa.bitwiseAnd(cirrus_bit_mask).eq(0)
        )
    )

    return image.updateMask(mask).divide(10000)


def get_data_for_point(point):
    """Obtiene datos ambientales para una observación de GLOBE."""

    # Extraer la fecha de la observación
    point_date = ee.Date(point.get("MeasuredDate"))
    day_before = point_date.advance(-1, "day")
    month_before = point_date.advance(-30, "day")

    # Temperatura de la superficie terrestre de MODIS de NASA
    lst = (
        ee.ImageCollection("MODIS/061/MOD11A1")
        .filterDate(month_before, point_date)
        .select([
            "LST_Day_1km",
            "QC_Day",
            "LST_Night_1km",
            "QC_Night",
            "Clear_day_cov",
            "Clear_night_cov"
        ])
        .median()
        # Obtener los valores en la ubicación del punto
        .reduceRegion(
            reducer=ee.Reducer.median(),
            geometry=point.geometry(),
            scale=1000
        )
    )
    point = point.set(lst)

    lst_value = point.get("LST_Day_1km").getInfo()
    point = point.set(
        "LST_Day_1km_F",
        to_fahrenheit(lst_value)
    )

    # Mediana de Sentinel-2 durante el mes anterior
    # Se utilizará para calcular el NDVI
    sentinel2 = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterDate(month_before, point_date)
        .map(mask_s2_clouds)
        .median()
        # Obtener la mediana de las bandas en la ubicación
        .reduceRegion(
            reducer=ee.Reducer.median(),
            geometry=point.geometry(),
            scale=10
        )
    )
    point = point.set(sentinel2)

    # Precipitación de Daymet V4 de NASA
    rain = (
        ee.ImageCollection("NASA/ORNL/DAYMET_V4")
        .filterDate(day_before, point_date)
        .select("prcp")  # Precipitación diaria total
        .sum()
        # Obtener la mediana en la ubicación del punto
        .reduceRegion(
            reducer=ee.Reducer.median(),
            geometry=point.geometry(),
            scale=1000
        )
    )
    point = point.set(rain)

    # Agregar información adicional
    b8 = ee.Number(point.get("B8"))
    b4 = ee.Number(point.get("B4"))
    point = point.set(
        "NDVI",
        b8.subtract(b4).divide(b8.add(b4))
    )

    return point

Utiliza esta función para obtener los datos de precipitación, temperatura y vegetación del punto seleccionado.

In [ ]:
point_fc = geemap.geopandas_to_ee(point).first()
result = get_data_for_point(point_fc)

Muestra los datos:

El índice de vegetación de diferencia normalizada, o NDVI, es un valor entre -1 y 1 que representa la cantidad de vegetación. Los valores altos, cercanos a 1, indican zonas con vegetación abundante, como bosques. Los valores bajos, cercanos a -1, indican zonas con poca vegetación.

In [ ]:
print(
    f"Observación de mosquitos de GLOBE en "
    f"{county_name}, {state_name}"
)
print(
    f"Fecha: {result.get('MeasuredDate').getInfo()}"
)
print(
    f"Fuente de agua: {result.get('WaterSource').getInfo()}"
)
print(
    "Precipitación: "
    f"{round(result.get('prcp').getInfo(), 2)} mm"
)
print(
    "Temperatura de la superficie terrestre: "
    f"{round(result.get('LST_Day_1km_F').getInfo(), 2)} °F"
)
print(
    "Índice de vegetación de diferencia normalizada (NDVI): "
    f"{round(result.get('NDVI').getInfo(), 2)}"
)

A continuación encontrarás más recursos de GLOBE y EMERGE:

- [Sitio web de EMERGE](https://geoemerge.com/) y [currículo](https://geo-di-lab.github.io/emerge-lessons)
- [Recursos para educadores de GLOBE](https://www.globe.gov/do-globe/resources/teaching-resources)
- [Biblioteca de recursos de GLOBE Mosquito Habitat Mapper](https://observer.globe.gov/do-globe-observer/mosquito-habitats/resource-library)
- [Biblioteca de recursos de GLOBE Land Cover](https://observer.globe.gov/do-globe-observer/land-cover)